# Overview

This notebook is intended to showcase some analytics that can be run on EV Charger time series data, with a focus on sumary statistics. It is not meant to be comprehensive, many further analyses could be envisioned.

The notebook is structured as follows:
- A Setup section for establishing a connection and performing basic data checks
- A Data Retrieval section for actual retrieval of the time series for EV Chargers as well as Meters
- Three Analysis sections:
    1. Comparing total load of EV Chargers with that of Meters over the entire period that data is available for
    2. Summarizing EV Charger load over the period by day and by month
    3. Visualizing temporal patterns in EV Charger load, e.g. versus day of the week, hour of the day or month of the year

These kinds of analyses can help utilities better understand the nature of EV Charger load, which in turn can lead to better planning and management of EV charging infrastructure or EV Charger rate design.

For more details about EV Charger usage analysis and how it can be done using Awesense's platform, please refer to the [UC09 - EV Charging and Use of Reserved Capacity Analysis](https://github.com/Awesense/edm-app-examples/blob/master/use_cases/usecase_descriptions/UC09%20-%20EV%20Charging%20and%20Use%20of%20Reserved%20Capacity%20Analysis.pdf) document.

# Setup

In [1]:
import getpass
import urllib.parse
import pandas as pd
import plotly.express as px
import numpy as np
import datetime

## Conection

In [ ]:
edm_address = getpass.getpass(prompt="EDM server address: ")

print("\nEDM login information")
edm_name = getpass.getpass(prompt="Username: ")
edm_password = getpass.getpass(prompt="Password: ")
edm_password = urllib.parse.quote(edm_password)

get_ipython().run_line_magic("load_ext", "sql")
get_ipython().run_line_magic(
    "sql", "postgresql://$edm_name:$edm_password@$edm_address/edm"
)
get_ipython().run_line_magic("config", "SqlMagic.displaycon = False")
get_ipython().run_line_magic("config", "SqlMagic.feedback = False")

# Delete the credential variables for security purposes.

del edm_name, edm_password


EDM login information


## Data checks

#### Checking the awefice grid is present

In [3]:
%%sql

SELECT * FROM grid;

grid_id,description,last_updated
awefice,awefice,2021-01-01 20:00:00+00:00
North Central Zone,North Central Zone,2021-01-01 20:00:00+00:00


In [ ]:
current_grid = "awefice"

#### Looking up the EV Chargers in this grid.

In [5]:
%%sql

SELECT *
FROM grid_element_data_source geds, grid_element ge
WHERE ge.grid_element_id = geds.grid_element_id
AND ge.grid_id = '{current_grid}'
AND ge.type = 'EVCharger';

grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction,grid_id_1,grid_element_id_1,type_1,customer_type,phases_1,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
a3dfe4ea-faaa-4a1c-a43e-b111d52e4684,awefice,evch_10,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_10,None,awefice,evch_10,EVCharger,ev_charger,ABC,None,True,True,False,None,con_135_t2,VCN.evch_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': None, 'label': 'evch_10', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_10', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'ChargePoint Home Flex [CPH50-NEMA14-50-L23]', 'rating_kva': '', 'max_current': 32, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_level': 'LV', 'charging_level': 'level_2', 'connector_type': 'NEMA 14-30', 'active_charging_power': 7680, 'parent_transformer_id': 'transformer_36', 'active_generation_power': 0}"
ab23affd-cc3d-43aa-9db7-13078ca38b14,awefice,evch_12,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_12,None,awefice,evch_12,EVCharger,ev_charger,ABC,None,True,True,False,None,con_152_t2,VCN.evch_12,both,None,0101000020E610000032415CF294C65EC0F99D2EC423A24840,"{'name': None, 'label': 'evch_12', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_12', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'Tesla Supercharger [Tesla]', 'rating_kva': '', 'max_current': 330, 'max_voltage': 410, 'is_dc_charger': True, 'voltage_level': 'LV', 'charging_level': 'level_3', 'connector_type': 'TESLA', 'active_charging_power': 135000, 'parent_transformer_id': 'transformer_47', 'active_generation_power': 0}"
d581fcac-42f1-4ecb-9a11-34735c71bf2f,awefice,evch_20,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_20,None,awefice,evch_20,EVCharger,ev_charger,ABC,None,True,True,False,None,con_228_t2,VCN.evch_20,both,None,0101000020E6100000B0AF7695AAC65EC0DC65CD8A58A24840,"{'name': None, 'label': 'evch_20', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_20', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'Grizzl-E EV Charger [GR1-14-24-P]', 'rating_kva': '', 'max_current': 16, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_level': 'LV', 'charging_level': 'level_2', 'connector_type': 'NEMA 14-30', 'active_charging_power': 3840, 'parent_transformer_id': 'transformer_63', 'active_generation_power': 0}"
bf63eabb-1b4a-4937-addc-77853cbc9171,awefice,evch_24,"[2021-01-01 08:00:00+00:00, None)",ABC,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_24,None,awefice,evch_24,EVCharger,ev_charger,ABC,None,True,True,False,None,con_312_t2,VCN.evch_24,both,None,0101000020E61000001C613FDF81C65EC0E40C592457A24840,"{'name': None, 'label': 'evch_24', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_24', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'ChargePoint CPE 200 [CPE200T-S-CHD-CMB]', 'rating_kva': '', 'max_current': 125, 'max_voltage': 500, 'is_dc_charger': True, 'voltage_level': 'LV', 'charging_level': 'level_3', 'connector_type': '[CHAdeMO,CCS1]', 'active_charging_power': 50000, 'parent_transformer_id': 'transformer_92', 'active_generation_power': 0}"
bc9a2651-736b-44e2-b3fd-e04463803a09,awefice,evch_3,"[2021-01-01 08:00:00+00:00, None)",A,TRUE_CONSUMPTION,awe_ev_charger,['kWh'],evch_3,None,awefice,evch_3,EVCharger,ev_charger,A,None,True,True,False,None,con_40_t2,VCN.evch_3,both,None,0101000020E61000005BCDA4FDBEC65EC0E0B44C793CA24840,"{'name': None, 'label': 'evch_3', 'sdp_id': None, 'latitude': None, 'meter_id': 'm_3', 'longitude': None, 'lv_feeder': None, 'ownership': 'private', 'make_model': 'Tesla Gen 3 Wall Connector [1457768-**-*]', 'rating_kva': '', 'max_current': 16, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_l

# Data Retrieval

In [6]:
# Defining Data Retrieval Function.

def _get_hourly_kwh_query(view_name, time_range):
    result = %sql SELECT ge_sources.grid_element_id, \
            date_trunc('hour', timestamp) at time zone 'America/Vancouver' as date_hour, \
            round(sum(value)::numeric, 2) AS "kWh", \
            ge_sources.type \
        FROM {view_name} ge_sources \
        JOIN ts_data_source_select(ge_sources.grid_element_data_source_id, 'kWh', {time_range}) tdss ON true \
        GROUP BY ge_sources.grid_element_id, date_hour, ge_sources.type;
    return result

def get_hourly_kwh(view_name: str, min_date: datetime.datetime=None, max_date: datetime.datetime=None,
                   chunk_duration: datetime.timedelta=None):
    """
    Retrieves the consumption (kWh) time series for grid elements as given by a view, aggregated per hour,
        converted from UTC to Vancouver timezone, and returns the final output as a dataframe.
    view_name must define a view of a subset of the grid_element_data_source table
    with grid_element_id, grid_element_data_source_id, valid (tstzrange), type columns

    Define min_date and max_date to limit the date range of the data returned and chunk_duration to query the database in chunks.

    Note that to avoid infinitesimally small load which impacts the capacity analysis a lot,
        the values are rounded to 2 decimal places.
    """
    element_years = %sql SELECT LOWER(valid) AS first_date, UPPER(valid) AS last_date \
        FROM {view_name};
    element_years = element_years.DataFrame()
    min_date = min_date or element_years['first_date'].min()
    max_date = max_date or element_years['last_date'].max()
    if pd.isna(max_date):
        max_date = datetime.datetime(datetime.datetime.now().year, 12, 31, tzinfo=min_date.tzinfo)

    chunked_data = []
    if chunk_duration is None:
        result = _get_hourly_kwh_query(view_name, time_range=f"TSTZRANGE('{min_date.isoformat()}', '{max_date.isoformat()}')")
        chunked_data.append(result.DataFrame())

    else:
        start_date = min_date
        while start_date < max_date:
            end_date = start_date + chunk_duration
            if end_date > max_date:
                end_date = max_date
            print(f"Fetching data from {start_date} to {end_date}")
            result = _get_hourly_kwh_query(view_name, time_range=f"TSTZRANGE('{start_date.isoformat()}', '{end_date.isoformat()}')")
            chunked_data.append(result.DataFrame())
            start_date = end_date

    df = pd.concat(chunked_data, ignore_index=True)
    df['kWh'] = df['kWh'].astype(float)
    return df

## Retrieving and summarizing EV Charger time series

In [7]:
%%sql CREATE OR REPLACE TEMPORARY VIEW ev_sources AS
    SELECT geds.grid_element_id, geds.grid_element_data_source_id, geds.valid, geds.type
    FROM grid_element ge
    JOIN grid_element_data_source geds
        ON ge.grid_id = geds.grid_id AND ge.grid_element_id = geds.grid_element_id
    WHERE ge.grid_id = '{current_grid}' AND ge.type = 'EVCharger';

[]

In [ ]:
## Depending on the size of your grid, if you want to read the data in chunks and iteratively build a data frame
## or only retrieve data in a certain period, you can do so by getting the hourly data for a certain duration at a time.
## However, this will likely be slower than querying the database for the full data at once.
from dateutil.relativedelta import relativedelta

df_evch = get_hourly_kwh("ev_sources", chunk_duration=relativedelta(months=6))
## For North Central Zone, may take 20 minutes or longer
df_evch = df_evch.rename(columns={"grid_element_id": "evch_id"}, inplace=False)

Fetching data from 2021-01-01 08:00:00+00:00 to 2021-07-01 08:00:00+00:00
Fetching data from 2021-07-01 08:00:00+00:00 to 2022-01-01 08:00:00+00:00
Fetching data from 2022-01-01 08:00:00+00:00 to 2022-07-01 08:00:00+00:00
Fetching data from 2022-07-01 08:00:00+00:00 to 2023-01-01 08:00:00+00:00
Fetching data from 2023-01-01 08:00:00+00:00 to 2023-07-01 08:00:00+00:00
Fetching data from 2023-07-01 08:00:00+00:00 to 2024-01-01 08:00:00+00:00
Fetching data from 2024-01-01 08:00:00+00:00 to 2024-07-01 08:00:00+00:00
Fetching data from 2024-07-01 08:00:00+00:00 to 2025-01-01 08:00:00+00:00
Fetching data from 2025-01-01 08:00:00+00:00 to 2025-07-01 08:00:00+00:00
Fetching data from 2025-07-01 08:00:00+00:00 to 2025-12-31 00:00:00+00:00


In [9]:
df_evch.head()

,evch_id,date_hour,kWh,type
0,evch_3,2021-06-12 07:00:00,0.00,TRUE_CONSUMPTION
1,evch_10,2021-04-27 00:00:00,0.00,TRUE_CONSUMPTION
2,evch_20,2021-02-25 06:00:00,0.00,TRUE_CONSUMPTION
3,evch_10,2021-02-14 14:00:00,7.68,TRUE_CONSUMPTION
4,evch_3,2021-05-30 16:00:00,0.00,TRUE_CONSUMPTION


In [10]:
# Summary stats for EV Chargers.
df_evch.describe()

,date_hour,kWh
count,218812,218812.000000
mean,2023-07-02 00:24:55.448147456,2.511900
min,2021-01-01 00:00:00,0.000000
25%,2022-04-01 22:00:00,0.000000
50%,2023-07-01 20:00:00,0.000000
75%,2024-09-29 17:00:00,0.000000
max,2025-12-30 15:00:00,73.210000
std,NaN,9.196193


## Retrieving and summarizing the Meter time series

In [11]:
%%sql CREATE OR REPLACE TEMPORARY VIEW meter_sources AS
    SELECT ge.grid_element_id AS evch_id, geds.grid_element_id,
        geds.grid_element_data_source_id, geds.valid, geds.type
    FROM grid_element ge
    JOIN grid_get_sources('{current_grid}', ge.grid_element_id, True) upstream_elements ON True
    JOIN grid_element_data_source geds
        ON ge.grid_id = geds.grid_id AND upstream_elements.grid_element_id = geds.grid_element_id
    WHERE ge.grid_id = '{current_grid}'
        AND ge.type='EVCharger' AND upstream_elements.type = 'Meter';

[]

In [ ]:
# Get time series consumption data for Meters, aggregated to hourly granularity.
df_meter = get_hourly_kwh("meter_sources")
# If grid_element_type is 'Meter', only retrieve data for meters with downstream EVChargers in the grid for efficiency
df_meter = df_meter.rename(columns={"grid_element_id": "meter_id"}, inplace=False)

In [13]:
df_meter.head()

,meter_id,date_hour,kWh,type
0,m_24,2022-07-30 16:00:00,28.51,PRODUCER
1,m_10,2022-12-31 11:00:00,5.26,CONSUMER
2,m_12,2025-05-21 23:00:00,2.45,CONSUMER
3,m_24,2021-12-17 17:00:00,0.00,PRODUCER
4,m_10,2025-11-07 00:00:00,0.00,PRODUCER


In [14]:
# Summary stats for Meters.
df_meter.describe()

,date_hour,kWh
count,306265,306265.000000
mean,2023-07-01 19:30:27.922876928,4.356972
min,2021-01-01 00:00:00,0.000000
25%,2022-04-01 20:00:00,0.000000
50%,2023-07-01 15:00:00,1.700000
75%,2024-09-29 10:00:00,4.260000
max,2025-12-30 15:00:00,79.430000
std,NaN,9.329119


## Further data munging (reshaping, joining Meters and EV Chargers)

In [ ]:
# Pivot the Meter data to separate out Producer (flow in) and Consumer (flow out) components.
df_meter_split = (
    df_meter.pivot(index=["meter_id", "date_hour"], columns="type", values="kWh")
    .rename_axis(None, axis=1)
    .reset_index()
)
if "PRODUCER" not in df_meter_split.columns:
    df_meter_split["PRODUCER"] = (
        np.nan
    )  # add PRODUCER column if not found in type column for any meter
df_meter_split.head()

,meter_id,date_hour,CONSUMER,PRODUCER
0,m_10,2021-01-01 00:00:00,4.65,0.0
1,m_10,2021-01-01 01:00:00,4.99,0.0
2,m_10,2021-01-01 02:00:00,5.17,0.0
3,m_10,2021-01-01 03:00:00,4.87,0.0
4,m_10,2021-01-01 04:00:00,5.56,0.0


In [ ]:
# Summary row counts
df_meter_split.groupby(["meter_id"]).count()

,date_hour,CONSUMER,PRODUCER
meter_id,,,
m_10,43742,43742,43742
m_12,43745,43745,0
m_20,43763,43763,0
m_24,43755,43755,43755
m_3,43763,43763,0


In [17]:
# Replace Nan values with 0.
df_meter_split.replace(np.nan, 0, inplace=True)
df_meter_split.head()

,meter_id,date_hour,CONSUMER,PRODUCER
0,m_10,2021-01-01 00:00:00,4.65,0.0
1,m_10,2021-01-01 01:00:00,4.99,0.0
2,m_10,2021-01-01 02:00:00,5.17,0.0
3,m_10,2021-01-01 03:00:00,4.87,0.0
4,m_10,2021-01-01 04:00:00,5.56,0.0


In [ ]:
# Compute the net of flow in and out for each meter.
df_meter_split["NET"] = df_meter_split["CONSUMER"] - df_meter_split["PRODUCER"]
df_meter_split.head()

,meter_id,date_hour,CONSUMER,PRODUCER,NET
0,m_10,2021-01-01 00:00:00,4.65,0.0,4.65
1,m_10,2021-01-01 01:00:00,4.99,0.0,4.99
2,m_10,2021-01-01 02:00:00,5.17,0.0,5.17
3,m_10,2021-01-01 03:00:00,4.87,0.0,4.87
4,m_10,2021-01-01 04:00:00,5.56,0.0,5.56


In [19]:
# Summary stats of Meter data.
df_meter_split.describe()

,date_hour,CONSUMER,PRODUCER,NET
count,218768,218768.000000,218768.000000,218768.000000
mean,2023-07-01 20:29:33.934030592,4.952775,1.146782,3.805993
min,2021-01-01 00:00:00,0.000000,0.000000,-46.400000
25%,2022-04-01 20:00:00,0.990000,0.000000,0.990000
50%,2023-07-01 15:00:00,2.290000,0.000000,2.290000
75%,2024-09-29 11:00:00,4.770000,0.000000,4.770000
max,2025-12-30 15:00:00,79.430000,46.400000,79.430000
std,NaN,9.699094,5.338505,11.572883


#### Retrieve the EV Charger to Meter correspondence from the DB:

In [20]:
%%sql evchs_meter <<

SELECT DISTINCT evch_id, grid_element_id as meter_id
    FROM meter_sources;

Returning data to local variable evchs_meter


In [21]:
# Convert to data frame for later use.
evchs_meter = evchs_meter.DataFrame()
evchs_meter

,evch_id,meter_id
0,evch_10,m_10
1,evch_12,m_12
2,evch_20,m_20
3,evch_24,m_24
4,evch_3,m_3


In [ ]:
# Add-in the information about the corresponding Meter in the EV Charger time series data frame.
df_evch_m = pd.merge(df_evch, evchs_meter, on="evch_id", how="inner")
df_evch_m.head()

,evch_id,date_hour,kWh,type,meter_id
0,evch_3,2021-06-12 07:00:00,0.00,TRUE_CONSUMPTION,m_3
1,evch_10,2021-04-27 00:00:00,0.00,TRUE_CONSUMPTION,m_10
2,evch_20,2021-02-25 06:00:00,0.00,TRUE_CONSUMPTION,m_20
3,evch_10,2021-02-14 14:00:00,7.68,TRUE_CONSUMPTION,m_10
4,evch_3,2021-05-30 16:00:00,0.00,TRUE_CONSUMPTION,m_3


# Analysis: Meter-EVCharger Correspondence

## Summing up data over all time

In [ ]:
# Print out the time ranges of the data for both Meters and EV Chargers for informational purposes
print(df_meter["date_hour"].min())
print(df_meter["date_hour"].max())
print(df_evch["date_hour"].min())
print(df_evch["date_hour"].max())

2021-01-01 00:00:00
2025-12-30 15:00:00
2021-01-01 00:00:00
2025-12-30 15:00:00


In [ ]:
# Compute the sum of EV Charger consumption from all the EV Chargers corresponding to each Meter.
meter_evch_kwh_portion = df_evch_m.groupby("meter_id").sum("kWh").reset_index()
meter_evch_kwh_portion

,meter_id,kWh
0,m_10,57477.56
1,m_12,343239.02
2,m_20,17338.42
3,m_24,114193.18
4,m_3,17385.61


In [ ]:
# Sum up the Meter time series (flow in, flow out and net) for each Meter.
meter_kwh_total_net = (
    df_meter_split.groupby("meter_id").sum(numeric_only=True).reset_index()
)
meter_kwh_total_net.sort_values(by="NET", ascending=False)

,meter_id,CONSUMER,PRODUCER,NET
1,m_12,523915.36,0.00,523915.36
0,m_10,204141.78,124.55,204017.23
4,m_3,79051.56,0.00,79051.56
2,m_20,68752.77,0.00,68752.77
3,m_24,207647.23,250754.76,-43107.53


In [ ]:
# Add in the information about respective EV Charger consumption for each Meter
meter_combined_data = pd.merge(
    meter_evch_kwh_portion, meter_kwh_total_net, on="meter_id", how="outer"
)
meter_combined_data = meter_combined_data.rename(
    columns={
        "kWh": "EV Charger Consumption",
        "PRODUCER": "Meter Flow In",
        "CONSUMER": "Meter Flow Out",
        "NET": "Meter Net",
    }
)
meter_combined_data

,meter_id,EV Charger Consumption,Meter Flow Out,Meter Flow In,Meter Net
0,m_10,57477.56,204141.78,124.55,204017.23
1,m_12,343239.02,523915.36,0.00,523915.36
2,m_20,17338.42,68752.77,0.00,68752.77
3,m_24,114193.18,207647.23,250754.76,-43107.53
4,m_3,17385.61,79051.56,0.00,79051.56


## Plotting

In [ ]:
# Plot per-Meter EVCharger consumption alongside Meter's Net flow, sorted by the latter descending.
fig = px.bar(
    meter_combined_data.sort_values(by="Meter Net", ascending=False),
    x="meter_id",
    y=["Meter Net", "EV Charger Consumption"],
    labels={"value": "kWh", "meter_id": "Meter"},
    barmode="group",
    color_discrete_map={"Meter Net": "gray", "EV Charger Consumption": "red"},
    title="<b>All-time</b> EV Charger Consumption compared to Meter Net<br>Meters sorted descending by their <i>Net</i>",
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Plot per-Meter EVCharger consumption alongside Meter's Net flow, sorted by the former descending.
fig = px.bar(
    meter_combined_data.sort_values(by="EV Charger Consumption", ascending=False),
    x="meter_id",
    y=["Meter Net", "EV Charger Consumption"],
    labels={"value": "kWh", "meter_id": "Meter"},
    barmode="group",
    color_discrete_map={"Meter Net": "gray", "EV Charger Consumption": "red"},
    title="<b>All-time</b> EV Charger Consumption compared to Meter Net<br>Meters sorted descending by their <i>downstream EV Charger Consumption</i>",
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Plot per-Meter EVCharger consumption alongside Meter's Flow In&Out, sorted by the former descending.
fig = px.bar(
    meter_combined_data.sort_values(by="EV Charger Consumption", ascending=False),
    x="meter_id",
    y=["Meter Flow Out", "Meter Flow In", "EV Charger Consumption"],
    labels={"value": "kWh", "meter_id": "Meter"},
    barmode="group",
    color_discrete_map={
        "Meter Flow Out": "blue",
        "Meter Flow In": "lightblue",
        "EV Charger Consumption": "red",
    },
    title="<b>All-time</b> EV Charger Consumption compared to Meter Flow In/Out<br>Meters sorted descending by their <i>downstream EV Charger Consumption</i>",
)
fig.update_xaxes(tickangle=45)
fig.show()

# Analysis: EV Charger Consumption Over All Time

## Helper functions

In [ ]:
def sum_loads(df, groupby_list, sum_list):
    """
    Returns a dataframe that is grouped by `groupby_list` columns and summed `sum_list` columns.
    Note that the `groupby_list` and `sum_list` may be a single string.
    """

    df_final = (
        df.groupby(groupby_list).sum(sum_list, min_count=1).reset_index()
    )  # min_count=1 keeps null as null in summation.

    return df_final

In [ ]:
def add_date_range(df, time_col, desc_str):
    """
    Returns a string with a date range added to `desc_str` string, to be used in a chart's title.
    """

    # Pull out min and max dates to use in the title.
    min_date = min(df[time_col].dt.date).strftime("%b %d, %Y")
    max_date = max(df[time_col].dt.date).strftime("%b %d, %Y")

    # Prepare the title.
    title = desc_str + " (" + min_date + " ~ " + max_date + ")"

    return title

## Daily EV Charger consumption over entire history

In [ ]:
# Create a copy to prepare the data with.
df = df_evch.copy()

# Pull out a date component out of timestamp to aggregate over.
df["date"] = df["date_hour"].dt.date

# Sum all EV Chargers loads for each date.
df_daily = sum_loads(df, "date", "kWh")

# Prepare a title with a date range.
title = add_date_range(
    df, time_col="date_hour", desc_str="<b>Daily</b> Sum of All EV Charger Load"
)

# Plot.
fig = px.line(
    df_daily,
    x="date",
    y="kWh",
    title=title,
    labels={"date": "Date"},
    width=1000,
    height=500,
    range_y=[0, max(df_daily["kWh"]) * 1.05],
)
fig.update_xaxes(tickangle=45)
fig.show()

## Monthly EV Charger consumption over entire history

In [ ]:
# Create a copy to prepare the data with.
df = df_evch.copy()

# Pull out a date component out of timestamp to aggregate over.
df["year_month"] = df["date_hour"].dt.strftime("%Y-%m")

# Sum EV Chargers for all parkades for each month.
df_monthly = sum_loads(df, "year_month", "kWh")

# Order the data by correct year month notion and not by alphabetically.
df_monthly = df_monthly.sort_values("year_month").reset_index()

# Prepare a title with a date range.
title = add_date_range(
    df, time_col="date_hour", desc_str="<b>Monthly</b> Sum of All EV Charger Load"
)

# Plot.
fig = px.bar(
    df_monthly,
    x="year_month",
    y="kWh",
    title=title,
    labels={"year_month": "Date"},
    width=1000,
    height=500,
    range_y=[0, max(df_monthly["kWh"]) * 1.05],
)
fig.update_xaxes(tickangle=45)
fig.show()

# Analysis: EV Charger consumption patterns w.r.t. various units of time

## Data processing and generic helper functions

In [ ]:
# Helper function
def enrich_data_for_plotting(df_evch):
    """
    Adds columns to the given dataframe for various units of time (e.g. hour-of-day, day-of-week, month)
    by computing them from the date_hour column of the given dataframe.
    """

    # Create a copy of the raw dataframe to work with.
    df_all = df_evch.copy()

    # Pull out year-week combinations to separate out each day based on the week.
    df_all.loc[:, "year_week"] = df_all["date_hour"].dt.strftime("%Y-%W")

    # Assign word version of days of the week.
    df_all.loc[:, "day"] = df_all["date_hour"].dt.day_name()

    # Assign numeric days of the week to maintain its ordinality.
    df_all.loc[:, "day_n"] = df_all["date_hour"].dt.weekday

    # Pull out the hours.
    df_all.loc[:, "hour"] = df_all["date_hour"].dt.hour

    # Pull out the months.
    df_all.loc[:, "month"] = df_all["date_hour"].dt.month_name()

    # Assign numeric month of year to maintain its ordinality.
    df_all.loc[:, "month_n"] = df_all["date_hour"].dt.month

    return df_all

In [35]:
# Enrich EV Charger data with hour-of-day/day-of-week/month information.
df_evch_enh = enrich_data_for_plotting(df_evch)
df_evch_enh.head()

,evch_id,date_hour,kWh,type,year_week,day,day_n,hour,month,month_n
0,evch_3,2021-06-12 07:00:00,0.00,TRUE_CONSUMPTION,2021-23,Saturday,5,7,June,6
1,evch_10,2021-04-27 00:00:00,0.00,TRUE_CONSUMPTION,2021-17,Tuesday,1,0,April,4
2,evch_20,2021-02-25 06:00:00,0.00,TRUE_CONSUMPTION,2021-08,Thursday,3,6,February,2
3,evch_10,2021-02-14 14:00:00,7.68,TRUE_CONSUMPTION,2021-06,Sunday,6,14,February,2
4,evch_3,2021-05-30 16:00:00,0.00,TRUE_CONSUMPTION,2021-21,Sunday,6,16,May,5


In [ ]:
# Helper function
def prep_data_for_overlaid_line_chart(df_evch_enh_agg, column, line_length):
    """
    Process data for overlaid line chart using the given column of the given dataframe
    and breaking the data into segments of line_length length (e.g. 7 for day-of-week chart,
    24 for hour-of-day chart).
    The easiest way to obtain a chart of overlaid lines is to insert rows with Nones into
    the dataframe at the places where a new segment is supposed to start.
    """
    df = df_evch_enh_agg.copy()

    # Order properly to then be able to remove first&last partial weeks.
    df.sort_values(
        ["year_week", "evch_id", column],
        ascending=True,
        inplace=True,
        ignore_index=True,
    )

    # find portion of first week (if it doesn't start on Monday)
    first_0 = next(idx for idx, val in enumerate(df[column], 1) if val == 0) - 1

    # find portion of last week (if it doesn't end on Sunday)
    last_6 = len(df) - next(
        idx for idx, val in enumerate(df.loc[::-1, column], 1) if val == 6
    )

    # remove first and last partial weeks
    df = df.iloc[first_0 : (last_6 + 1),].reset_index(drop=True)

    # Re-sort by evch_id first
    df.sort_values(
        ["evch_id", "year_week", column],
        ascending=True,
        inplace=True,
        ignore_index=True,
    )

    # Get the distinct evch_ids.
    evch_ids = df["evch_id"].unique()
    evch_ids_and_counts = (
        df.groupby("evch_id")["kWh"]
        .count()
        .reset_index()
        .sort_values("evch_id", ascending=True)
    )

    # Insert Nones with proper evch_id for plotting purposes (to break into segments)
    idx = 0
    for i in range(len(evch_ids_and_counts)):
        evch_id_val, count = evch_ids_and_counts.iloc[i, :]
        n_weeks = count // line_length
        break_rows = pd.DataFrame(
            {"hour": np.NaN, "kWh": None, "evch_id": evch_id_val, "year_week": None},
            index=[idx + line_length * (i + 1) - 0.5 for i in range(n_weeks)],
        )
        break_rows = break_rows.dropna(axis=1, how="all")
        df = pd.concat([df, break_rows], ignore_index=False)
        idx = idx + count
    # Get the Nones to be in the right spots
    df = df.sort_index().reset_index(drop=True)

    return df

## Day-of-Week Analysis

In [ ]:
# Helper function
def plot_evch_daily_load_vs_dow(df_all, evch_id, chart_type):
    """
    Makes a plot of EV Charger Load vs. Day of Week.
    Can plot one EV Charger or many (if evch_id='all'), in either multi-line form or boxplot form
    as specified by chart_type.
    """

    # Prepare a title with an appropriate date range.
    df_for_title = df_all if evch_id == "all" else df_all[df_all["evch_id"] == evch_id]
    title = add_date_range(
        df_for_title,
        time_col="date_hour",
        desc_str="EV Charger <b>Daily</b> Sum of Load vs. <b>Day-of-week</b>"
        + "<br>"
        + (
            "<b>All</b> EV Chargers"
            if evch_id == "all"
            else "EV Charger: <b>" + evch_id + "</b>"
        ),
    )

    # Sum EV Load data to days.
    df_agg = sum_loads(
        df_all, ["evch_id", "year_week", "day", "day_n", "month", "month_n"], "kWh"
    )

    # Filter out for a single evch id if needed.
    if evch_id == "all":
        df = df_agg
    else:
        df = df_agg[df_agg["evch_id"] == evch_id].copy()

    # Plot appropriate chart based on the chart type specified.
    if chart_type == "box":
        # Order by numeric days of the week to order the plot properly.
        df.sort_values("day_n", ascending=True, inplace=True)
        # Plot.
        fig = px.box(
            df,
            x="day",
            y="kWh",
            title=title,
            labels={"day": "Day of Week"},
            width=1000,
            height=500,
        )
        fig.update_traces(marker_color="red")

    elif chart_type == "line":
        # Split into weekly chunks each with 7 daily values
        df = prep_data_for_overlaid_line_chart(df, "day_n", 7)

        # Plot.
        fig = px.line(
            df,
            x="day",
            y="kWh",
            color="evch_id",
            title=title,
            labels={"day": "Day of Week", "evch_id": "EV Charger ID"},
            width=1000,
            height=500,
        )
        # Set the multiple lines to be more transparent
        fig.update_traces(opacity=0.2)

    fig.show()

In [ ]:
# Plotting day-of-week boxplots and line charts, first for all EV Chargers combined and then one EV Charger per plot.
plot_evch_daily_load_vs_dow(df_evch_enh, evch_id="all", chart_type="line")
plot_evch_daily_load_vs_dow(df_evch_enh, evch_id="all", chart_type="box")

In [ ]:
evch_id = input("Enter evch_id to plot individual EVCharger: ")
plot_evch_daily_load_vs_dow(df_evch_enh, evch_id, chart_type="line")
plot_evch_daily_load_vs_dow(df_evch_enh, evch_id, chart_type="box")

## Hour-of-Day Analysis

In [ ]:
def plot_evch_hourly_load_vs_hod(df_all, evch_id, chart_type):
    """
    Makes a plot of EV Charger Load vs. Hour-of-Day.
    Can plot one EV Charger or many, in either multi-line form or boxplot form
    """

    # Prepare a title with an appropriate date range.
    df_for_title = df_all if evch_id == "all" else df_all[df_all["evch_id"] == evch_id]
    title = add_date_range(
        df_for_title,
        time_col="date_hour",
        desc_str="EV Charger <b>Hourly</b> Load vs. <b>Hour-of-Day</b>"
        + "<br>"
        + (
            "<b>All</b> EV Chargers"
            if evch_id == "all"
            else "EV Charger: <b>" + evch_id + "</b>"
        ),
    )

    # Sum EV Load data to hours.
    df_agg = sum_loads(df_all, ["evch_id", "year_week", "hour"], "kWh")

    # Filter out for a single evch id if needed.
    if evch_id == "all":
        df = df_agg
    else:
        df = df_agg[df_agg["evch_id"] == evch_id].copy()

    # Plot appropriate chart based on the chart type specified.
    if chart_type == "box":
        # Order by numeric days of the week to order the plot properly.
        df.sort_values("hour", ascending=True, inplace=True)

        fig = px.box(
            df,
            x="hour",
            y="kWh",
            title=title,
            labels={"hour": "Hour of Day"},
            width=1000,
            height=500,
        )
        fig.update_traces(marker_color="red")

    elif chart_type == "line":
        # Split into daily chunks each with 24 hourly values
        df = prep_data_for_overlaid_line_chart(df, "hour", 24)

        # Plot.
        fig = px.line(
            df,
            x="hour",
            y="kWh",
            color="evch_id",
            title=title,
            labels={"hour": "Hour of Day", "evch_id": "EV Charger ID"},
            width=1000,
            height=500,
        )
        # Set the multiple lines to be more transparent
        fig.update_traces(opacity=0.2)

    fig.show()

In [ ]:
# Plotting hour-of-day boxplots and line charts, first for all EV Chargers combined and then one EV Charger per plot.
plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id="all", chart_type="line")
plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id="all", chart_type="box")

In [ ]:
evch_id = input("Enter evch_id to plot individual EVCharger: ")
plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id, chart_type="line")
plot_evch_hourly_load_vs_hod(df_evch_enh, evch_id, chart_type="box")

## Month-of-Year Analysis

In [ ]:
def plot_evch_daily_load_vs_moy(df_all, evch_id):
    """
    Makes a plot of Daily EV Charger Load vs. Month of Year.
    Can plot one EV Charger or many, but in boxplot form only
    """

    # Prepare a title with an appropriate date range.
    df_for_title = df_all if evch_id == "all" else df_all[df_all["evch_id"] == evch_id]
    title = add_date_range(
        df_for_title,
        time_col="date_hour",
        desc_str="EV Charger <b>Daily</b> Sum of Load vs. <b>Month-of-Year</b>"
        + "<br>"
        + (
            "<b>All</b> EV Chargers"
            if evch_id == "all"
            else "EV Charger: <b>" + evch_id + "</b>"
        ),
    )

    # Sum EV Load data to days.
    df_agg = sum_loads(df_all, ["evch_id", "year_week", "month", "month_n"], "kWh")

    # Filter out for a single evch id if needed.
    if evch_id == "all":
        df = df_agg
    else:
        df = df_agg[df_agg["evch_id"] == evch_id].copy()

    # Order by numeric days of the week to order the plot properly.
    df.sort_values("month_n", ascending=True, inplace=True)

    # Plot.
    fig = px.box(
        df,
        x="month",
        y="kWh",
        title=title,
        labels={"month": "Month"},
        width=1000,
        height=500,
    )
    fig.update_traces(marker_color="red")

    fig.show(config={"staticPlot": True})

In [ ]:
plot_evch_daily_load_vs_moy(df_evch_enh, evch_id="all")

In [ ]:
evch_id = input("Enter evch_id to plot individual EVCharger: ")
plot_evch_daily_load_vs_moy(df_evch_enh, evch_id)